In [2]:
# ==========================================
# IMPORTS DIPERBAIKI
# ==========================================
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1" 
os.environ["MKL_NUM_THREADS"] = "1" 
os.environ["VECLIB_MAXIMUM_THREADS"] = "1" 
os.environ["NUMEXPR_NUM_THREADS"] = "1" 
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.signal import butter, filtfilt, welch
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, recall_score, precision_score, 
    roc_curve, auc, roc_auc_score, matthews_corrcoef
)
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, BatchNormalization,
    Dense, Dropout, LSTM, Bidirectional,
    LeakyReLU, GlobalAveragePooling1D, Flatten,
    Add, LayerNormalization, GRU, GlobalMaxPooling1D,
    Concatenate, Reshape, Multiply, Lambda,
    UpSampling1D, Conv1DTranspose, Attention
)
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l1_l2
import gc
import warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm

print("TensorFlow Version:", tf.__version__)
print("NumPy Version:", np.__version__)

# ==========================================
# 1. SET SEED UNTUK REPRODUCIBILITY
# ==========================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ==========================================
# 2. LOAD DATA YANG SUDAH ADA
# ==========================================
print("\n" + "="*80)
print("🚀 LOADING DATA")
print("="*80)

# Load atau buat data sederhana jika tidak ada
try:
    X_raw = np.load('/kaggle/input/ecg-processed/X_raw.npy')
    y_raw = np.load('/kaggle/input/ecg-processed/y_raw.npy')
    patient_ids = np.load('/kaggle/input/ecg-processed/patient_ids.npy')
    encoder = joblib.load('/kaggle/input/ecg-processed/label_encoder.joblib')
    print("✅ Loaded existing processed data")
except:
    print("⚠️  No processed data found, using sample data...")
    # Create sample data for testing
    n_samples = 5000
    seq_len = 1000
    n_channels = 12
    X_raw = np.random.randn(n_samples, seq_len, n_channels).astype(np.float32)
    y_raw = np.random.randint(0, 4, n_samples)
    patient_ids = np.random.randint(0, 100, n_samples)
    
    # Create encoder
    encoder = LabelEncoder()
    encoder.classes_ = np.array(['Left_Ventricular_Region', 'Outflow_Tract_Region', 
                                'Right_Ventricular_Region', 'Septal_Region'])
    print("✅ Created sample data for testing")

class_names = encoder.classes_
num_classes = len(class_names)

print(f"\n✅ Data ready:")
print(f"  Total samples: {X_raw.shape[0]}")
print(f"  Sequence length: {X_raw.shape[1]}")
print(f"  Channels: {X_raw.shape[2]}")
print(f"  Number of patients: {len(np.unique(patient_ids))}")
print(f"  Number of classes: {num_classes}")
print(f"  Classes: {list(class_names)}")

# ==========================================
# 3. ATTENTION LAYER YANG DIPERBAIKI
# ==========================================
class AttentionLayer(tf.keras.layers.Layer):
    """Custom Attention Layer untuk Keras"""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    
    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                initializer="zeros")
        super(AttentionLayer, self).build(input_shape)
    
    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)
    
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

# ==========================================
# 4. CNN-BiLSTM-ATTENTION MODEL YANG DIPERBAIKI
# ==========================================
def build_cnn_bilstm_attention_model(input_shape, num_classes):
    """CNN-BiLSTM dengan Attention Mechanism yang diperbaiki"""
    
    inputs = Input(shape=input_shape, name='input_layer')
    
    # CNN Feature Extractor
    x = Conv1D(32, 7, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.1)(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)
    
    x = Conv1D(64, 5, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.1)(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.3)(x)
    
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.1)(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.3)(x)
    
    # Bidirectional LSTM
    x = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2))(x)
    
    # Attention Mechanism menggunakan custom layer
    attention_probs = Dense(1, activation='tanh')(x)
    attention_probs = tf.keras.layers.Softmax(axis=1)(attention_probs)
    attention_mul = Multiply()([x, attention_probs])
    
    # Global Pooling
    x = GlobalAveragePooling1D()(attention_mul)
    
    # Dense Layers
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    # Output Layer
    outputs = Dense(num_classes, activation='softmax', name='classification_output')(x)
    
    model = Model(inputs, outputs, name='CNN_BiLSTM_Attention')
    
    return model

# ==========================================
# 5. ECG GAN YANG DIPERBAIKI (SHAPE MATCH)
# ==========================================
def build_ecg_gan(input_shape, latent_dim=100):
    """ECG GAN untuk data augmentation yang diperbaiki"""
    
    seq_len, n_channels = input_shape
    
    # Generator dengan output shape yang tepat
    generator_input = Input(shape=(latent_dim,))
    
    # Pastikan output shape sesuai dengan input_shape
    # Hitung dimensi setelah upsampling
    target_len = seq_len
    initial_len = target_len // 16  # Karena 4x upsampling dengan faktor 2
    
    x = Dense(128 * initial_len)(generator_input)
    x = Reshape((initial_len, 128))(x)
    
    # Upsampling bertahap
    x = Conv1DTranspose(64, 5, padding='same', activation='relu')(x)
    x = UpSampling1D(2)(x)
    
    x = Conv1DTranspose(32, 5, padding='same', activation='relu')(x)
    x = UpSampling1D(2)(x)
    
    x = Conv1DTranspose(16, 5, padding='same', activation='relu')(x)
    x = UpSampling1D(2)(x)
    
    x = Conv1DTranspose(8, 5, padding='same', activation='relu')(x)
    x = UpSampling1D(2)(x)
    
    # Final layer dengan padding 'same' untuk memastikan shape benar
    generator_output = Conv1D(n_channels, 5, padding='same', activation='tanh', 
                            name='generator_output')(x)
    
    # Jika shape masih belum tepat, gunakan cropping atau padding
    if generator_output.shape[1] > seq_len:
        generator_output = generator_output[:, :seq_len, :]
    elif generator_output.shape[1] < seq_len:
        pad_len = seq_len - generator_output.shape[1]
        generator_output = tf.keras.layers.ZeroPadding1D((0, pad_len))(generator_output)
    
    generator = Model(generator_input, generator_output, name='Generator')
    
    # Discriminator
    discriminator_input = Input(shape=input_shape)
    
    x = Conv1D(32, 5, padding='same', activation='relu')(discriminator_input)
    x = LeakyReLU(alpha=0.2)(x)
    x = MaxPooling1D(2)(x)
    
    x = Conv1D(64, 5, padding='same', activation='relu')(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = MaxPooling1D(2)(x)
    
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = GlobalAveragePooling1D()(x)
    
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    discriminator_output = Dense(1, activation='sigmoid', name='discriminator_output')(x)
    
    discriminator = Model(discriminator_input, discriminator_output, name='Discriminator')
    
    return generator, discriminator

def train_ecg_gan(generator, discriminator, X_train, epochs=50, batch_size=32, latent_dim=100):
    """Train ECG GAN"""
    print(f"\n🎨 Training ECG GAN for data augmentation...")
    
    # Compile discriminator
    discriminator.compile(
        optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Combined model (GAN)
    discriminator.trainable = False
    gan_input = Input(shape=(latent_dim,))
    generated_ecg = generator(gan_input)
    gan_output = discriminator(generated_ecg)
    
    gan = Model(gan_input, gan_output, name='GAN')
    gan.compile(
        optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
        loss='binary_crossentropy'
    )
    
    # Rescale data ke range [-1, 1] untuk GAN
    X_train_gan = (X_train - X_train.min()) / (X_train.max() - X_train.min()) * 2 - 1
    
    # Training
    real_labels = np.ones((batch_size, 1))
    fake_labels = np.zeros((batch_size, 1))
    
    for epoch in range(epochs):
        # Train discriminator
        idx = np.random.randint(0, X_train_gan.shape[0], batch_size)
        real_ecgs = X_train_gan[idx]
        
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        fake_ecgs = generator.predict(noise, verbose=0)
        
        d_loss_real = discriminator.train_on_batch(real_ecgs, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_ecgs, fake_labels)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        
        # Train generator
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = gan.train_on_batch(noise, real_labels)
        
        if epoch % 10 == 0:
            print(f"  Epoch {epoch}: D_loss = {d_loss[0]:.4f}, D_acc = {d_loss[1]:.4f}, G_loss = {g_loss:.4f}")
    
    print("✅ ECG GAN training completed")
    return generator

def augment_with_gan(generator, X_train, y_train, n_samples_per_class=500, latent_dim=100):
    """Augment data menggunakan trained GAN"""
    print(f"\n🔧 Augmenting data with GAN...")
    
    unique_classes = np.unique(y_train)
    X_augmented = [X_train]
    y_augmented = [y_train]
    
    for class_idx in unique_classes:
        class_mask = y_train == class_idx
        X_class = X_train[class_mask]
        n_current = len(X_class)
        
        if n_current < n_samples_per_class:
            n_needed = n_samples_per_class - n_current
            n_batches = (n_needed // 32) + 1
            
            for i in range(n_batches):
                batch_size = min(32, n_needed - i*32)
                if batch_size <= 0:
                    break
                    
                noise = np.random.normal(0, 1, (batch_size, latent_dim))
                generated_ecg = generator.predict(noise, verbose=0)
                
                # Add small noise untuk realism
                generated_ecg += 0.02 * np.random.randn(*generated_ecg.shape)
                
                X_augmented.append(generated_ecg)
                y_augmented.extend([class_idx] * batch_size)
    
    X_augmented = np.vstack(X_augmented)
    y_augmented = np.hstack(y_augmented)
    
    # Shuffle
    idx = np.random.permutation(len(X_augmented))
    X_augmented = X_augmented[idx]
    y_augmented = y_augmented[idx]
    
    print(f"  Original samples: {len(X_train)}")
    print(f"  Augmented samples: {len(X_augmented)}")
    print(f"  Total samples after GAN: {len(X_augmented)}")
    
    return X_augmented, y_augmented

# ==========================================
# 6. TRAINING PIPELINE YANG DIPERBAIKI
# ==========================================
def train_with_gan_augmentation(X_train, y_train, X_val, y_val, input_shape, num_classes):
    """Train dengan GAN augmentation yang diperbaiki"""
    
    print(f"\n⚡ Training with GAN augmentation...")
    
    try:
        # Build and train GAN
        generator, discriminator = build_ecg_gan(input_shape)
        
        # Cek shape
        print(f"  Input shape: {input_shape}")
        print(f"  Generator output shape: {generator.output_shape}")
        print(f"  Discriminator input shape: {discriminator.input_shape}")
        
        trained_generator = train_ecg_gan(generator, discriminator, X_train, epochs=30)
        
        # Augment data dengan GAN
        X_train_aug, y_train_aug = augment_with_gan(trained_generator, X_train, y_train, 
                                                   n_samples_per_class=1000)
        
    except Exception as e:
        print(f"  ⚠️  GAN training failed: {e}")
        print(f"  🔄 Using SMOTE-like augmentation instead...")
        
        # Fallback: Simple oversampling jika GAN gagal
        from imblearn.over_sampling import RandomOverSampler
        
        # Reshape untuk oversampling
        X_train_reshaped = X_train.reshape(X_train.shape[0], -1)
        ros = RandomOverSampler(random_state=SEED)
        X_train_aug_reshaped, y_train_aug = ros.fit_resample(X_train_reshaped, y_train)
        X_train_aug = X_train_aug_reshaped.reshape(-1, X_train.shape[1], X_train.shape[2])
        
        print(f"  Samples after oversampling: {len(X_train_aug)}")
    
    # Build classification model
    model = build_cnn_bilstm_attention_model(input_shape, num_classes)
    
    # Compile
    optimizer = Adam(learning_rate=0.001)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    
    # Callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        )
    ]
    
    # Class weights
    try:
        class_weights = class_weight.compute_class_weight(
            'balanced',
            classes=np.unique(y_train_aug),
            y=y_train_aug
        )
        class_weights = dict(enumerate(class_weights))
    except:
        class_weights = None
    
    # Train
    print(f"\n🏋️ Training classification model...")
    start_time = time.time()
    
    history = model.fit(
        X_train_aug, to_categorical(y_train_aug, num_classes),
        validation_data=(X_val, to_categorical(y_val, num_classes)),
        epochs=30,
        batch_size=32,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1
    )
    
    train_time = time.time() - start_time
    
    # Evaluate
    y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='weighted')
    
    print(f"\n✅ Training completed in {train_time:.1f}s")
    print(f"   Accuracy: {acc:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    
    return model, history, acc, f1

# ==========================================
# 7. SIMPLE TRAINING PIPELINE (TANPA GAN)
# ==========================================
def train_without_gan(X_train, y_train, X_val, y_val, input_shape, num_classes):
    """Training tanpa GAN (fallback)"""
    
    print(f"\n⚡ Training without GAN (simple model)...")
    
    # Handle class imbalance dengan oversampling
    from imblearn.over_sampling import RandomOverSampler
    
    X_train_reshaped = X_train.reshape(X_train.shape[0], -1)
    ros = RandomOverSampler(random_state=SEED)
    X_train_bal_reshaped, y_train_bal = ros.fit_resample(X_train_reshaped, y_train)
    X_train_bal = X_train_bal_reshaped.reshape(-1, X_train.shape[1], X_train.shape[2])
    
    print(f"  Original samples: {len(X_train)}")
    print(f"  Balanced samples: {len(X_train_bal)}")
    
    # Build model
    model = build_cnn_bilstm_attention_model(input_shape, num_classes)
    
    # Compile
    optimizer = Adam(learning_rate=0.001)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        )
    ]
    
    # Train
    print(f"\n🏋️ Training classification model...")
    start_time = time.time()
    
    history = model.fit(
        X_train_bal, to_categorical(y_train_bal, num_classes),
        validation_data=(X_val, to_categorical(y_val, num_classes)),
        epochs=30,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )
    
    train_time = time.time() - start_time
    
    # Evaluate
    y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='weighted')
    
    print(f"\n✅ Training completed in {train_time:.1f}s")
    print(f"   Accuracy: {acc:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    
    return model, history, acc, f1

# ==========================================
# 8. CROSS-VALIDATION YANG DIPERBAIKI
# ==========================================
def run_cross_validation(X, y, patient_ids, class_names, num_classes, n_folds=3, use_gan=True):
    """Cross-validation dengan/setelah GAN"""
    
    print("\n" + "="*80)
    print("🧪 CROSS-VALIDATION" + (" WITH ECG GAN" if use_gan else ""))
    print("="*80)
    
    # Group by patient
    unique_patients = np.unique(patient_ids)
    patient_to_indices = {}
    
    for pid in unique_patients:
        patient_to_indices[pid] = np.where(patient_ids == pid)[0]
    
    # Patient labels
    patient_labels = []
    for pid in unique_patients:
        idx = patient_to_indices[pid]
        patient_labels.append(np.bincount(y[idx]).argmax())
    
    patient_labels = np.array(patient_labels)
    
    # Stratified KFold
    n_folds = min(n_folds, len(unique_patients))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    
    fold_results = []
    best_model = None
    best_acc = 0
    
    fold_no = 1
    
    for train_patient_idx, test_patient_idx in skf.split(unique_patients, patient_labels):
        print(f"\n\n{'='*60}")
        print(f"🎯 FOLD {fold_no} / {n_folds}")
        print(f"{'='*60}")
        
        # Split patients
        train_patients = unique_patients[train_patient_idx]
        test_patients = unique_patients[test_patient_idx]
        
        # Get indices
        train_indices = np.concatenate([patient_to_indices[pid] for pid in train_patients])
        test_indices = np.concatenate([patient_to_indices[pid] for pid in test_patients])
        
        X_train = X[train_indices]
        y_train = y[train_indices]
        X_test = X[test_indices]
        y_test = y[test_indices]
        
        print(f"  Train: {len(train_patients)} patients, {len(X_train)} beats")
        print(f"  Test: {len(test_patients)} patients, {len(X_test)} beats")
        
        # Standardize
        scaler = StandardScaler()
        X_train_2d = X_train.reshape(-1, X_train.shape[-1])
        X_test_2d = X_test.reshape(-1, X_test.shape[-1])
        
        X_train_scaled_2d = scaler.fit_transform(X_train_2d)
        X_test_scaled_2d = scaler.transform(X_test_2d)
        
        X_train_scaled = X_train_scaled_2d.reshape(X_train.shape)
        X_test_scaled = X_test_scaled_2d.reshape(X_test.shape)
        
        # Train model
        if use_gan:
            try:
                model, history, acc, f1 = train_with_gan_augmentation(
                    X_train_scaled, y_train,
                    X_test_scaled, y_test,
                    input_shape=X_train_scaled.shape[1:],
                    num_classes=num_classes
                )
            except Exception as e:
                print(f"  ⚠️  GAN failed, using simple training: {e}")
                model, history, acc, f1 = train_without_gan(
                    X_train_scaled, y_train,
                    X_test_scaled, y_test,
                    input_shape=X_train_scaled.shape[1:],
                    num_classes=num_classes
                )
        else:
            model, history, acc, f1 = train_without_gan(
                X_train_scaled, y_train,
                X_test_scaled, y_test,
                input_shape=X_train_scaled.shape[1:],
                num_classes=num_classes
            )
        
        # Store results
        fold_results.append({
            'Fold': fold_no,
            'Accuracy': acc,
            'F1_Score': f1,
            'Train_Patients': len(train_patients),
            'Test_Patients': len(test_patients),
            'Train_Samples': len(X_train),
            'Test_Samples': len(X_test)
        })
        
        # Save best model
        if acc > best_acc:
            best_acc = acc
            best_model = model
            best_X_test = X_test_scaled.copy()
            best_y_test = y_test.copy()
            best_history = history
            print(f"    🏆 New best model! Accuracy: {acc:.4f}")
        
        fold_no += 1
        
        # Cleanup
        tf.keras.backend.clear_session()
        gc.collect()
    
    # Results
    df_results = pd.DataFrame(fold_results)
    
    print(f"\n{'='*60}")
    print(f"📊 FINAL RESULTS" + (" (WITH GAN)" if use_gan else " (NO GAN)"))
    print(f"{'='*60}")
    print(f"Average Accuracy: {df_results['Accuracy'].mean():.4f} ± {df_results['Accuracy'].std():.4f}")
    print(f"Best Accuracy: {df_results['Accuracy'].max():.4f}")
    print(f"Average F1-Score: {df_results['F1_Score'].mean():.4f} ± {df_results['F1_Score'].std():.4f}")
    
    return df_results, best_model, best_X_test, best_y_test, best_history

# ==========================================
# 9. VISUALIZATION & REPORTING (SAMA)
# ==========================================
def create_comprehensive_report(df_results, model, X_test, y_test, class_names, history):
    """Create comprehensive report"""
    
    print("\n" + "="*80)
    print("📊 CREATING COMPREHENSIVE REPORT")
    print("="*80)
    
    # 1. Performance Summary
    print("\n📋 PERFORMANCE SUMMARY")
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    
    # Calculate metrics
    y_pred_proba = model.predict(X_test, verbose=0)
    y_test_bin = label_binarize(y_test, classes=range(num_classes))
    
    summary_df = pd.DataFrame({
        'Metric': ['Accuracy', 'F1-Score', 'Precision', 'Recall', 
                  'AUC', 'Matthews Correlation'],
        'Value': [
            f"{accuracy_score(y_test, y_pred):.4f}",
            f"{f1_score(y_test, y_pred, average='weighted'):.4f}",
            f"{precision_score(y_test, y_pred, average='weighted'):.4f}",
            f"{recall_score(y_test, y_pred, average='weighted'):.4f}",
            f"{roc_auc_score(y_test_bin, y_pred_proba, average='weighted', multi_class='ovr'):.4f}",
            f"{matthews_corrcoef(y_test, y_pred):.4f}"
        ]
    })
    print(summary_df)
    summary_df.to_csv("performance_summary.csv", index=False)
    
    # 2. Confusion Matrix
    print("\n📊 CONFUSION MATRIX")
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 3. Learning Curves
    print("\n📊 LEARNING CURVES")
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history.history['accuracy'], label='Train', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Val', linewidth=2)
    plt.title('Accuracy', fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 2)
    plt.plot(history.history['loss'], label='Train', linewidth=2)
    plt.plot(history.history['val_loss'], label='Val', linewidth=2)
    plt.title('Loss', fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 3)
    if 'auc' in history.history:
        plt.plot(history.history['auc'], label='Train', linewidth=2)
        plt.plot(history.history['val_auc'], label='Val', linewidth=2)
        plt.title('AUC', fontweight='bold')
        plt.xlabel('Epoch')
        plt.ylabel('AUC')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('learning_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 4. Classification Report
    print("\n📋 DETAILED CLASSIFICATION REPORT")
    report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    print(report_df)
    report_df.to_csv("detailed_classification_report.csv")
    
    print("\n✅ Comprehensive report created!")

# ==========================================
# 10. MAIN EXECUTION YANG DIPERBAIKI
# ==========================================
if __name__ == "__main__":
    
    print("\n" + "="*80)
    print("🧬 CNN-BiLSTM-ATTENTION WITH ECG GAN FOR IMBALANCED DATA")
    print("="*80)
    
    # Check GPU
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"✅ GPU available: {len(gpus)} device(s)")
        # Set memory growth
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print("⚠️  No GPU available, using CPU")
    
    # Pilihan: Gunakan GAN atau tidak
    USE_GAN = True  # Ganti ke False jika ingin tanpa GAN
    
    try:
        # Run cross-validation
        df_results, best_model, X_test_best, y_test_best, best_history = run_cross_validation(
            X_raw, y_raw, patient_ids, class_names, num_classes, 
            n_folds=3, use_gan=USE_GAN
        )
        
        # Create comprehensive report
        create_comprehensive_report(df_results, best_model, X_test_best, y_test_best, 
                                  class_names, best_history)
        
        # Save the best model
        model_name = 'best_cnn_bilstm_attention_with_gan.h5' if USE_GAN else 'best_cnn_bilstm_attention_no_gan.h5'
        best_model.save(model_name)
        print(f"\n💾 Model saved: {model_name}")
        
        # Save results
        results_name = 'cv_results_with_gan.csv' if USE_GAN else 'cv_results_no_gan.csv'
        df_results.to_csv(results_name, index=False)
        
        # Final summary
        print("\n" + "="*80)
        print("🎉 EXPERIMENT COMPLETED SUCCESSFULLY!")
        print("="*80)
        
        print(f"\n🏆 FINAL RESULTS:")
        print(f"   Average Accuracy: {df_results['Accuracy'].mean():.4f}")
        print(f"   Best Accuracy: {df_results['Accuracy'].max():.4f}")
        print(f"   Average F1-Score: {df_results['F1_Score'].mean():.4f}")
        
        print(f"\n📁 OUTPUT FILES:")
        files = [
            'performance_summary.csv',
            'detailed_classification_report.csv',
            results_name,
            'confusion_matrix.png',
            'learning_curves.png'
        ]
        
        for i, file in enumerate(files, 1):
            if os.path.exists(file):
                size = os.path.getsize(file) / 1024
                print(f"   {i:2d}. {file:45} ({size:.1f} KB)")
        
        print(f"\n🔧 ARCHITECTURE SUMMARY:")
        print(f"   • CNN Feature Extractor (3 layers)")
        print(f"   • Bidirectional LSTM for temporal features")
        print(f"   • Attention Mechanism for focus")
        if USE_GAN:
            print(f"   • ECG GAN for data augmentation (imbalance handling)")
        else:
            print(f"   • Random Oversampling for imbalance handling")
        print(f"   • Patient-wise cross-validation")
        
    except Exception as e:
        print(f"\n❌ ERROR in main execution: {e}")
        import traceback
        traceback.print_exc()
        
        print(f"\n🔄 Trying simple training without cross-validation...")
        
        # Simple train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=SEED
        )
        
        # Standardize
        scaler = StandardScaler()
        X_train_2d = X_train.reshape(-1, X_train.shape[-1])
        X_test_2d = X_test.reshape(-1, X_test.shape[-1])
        
        X_train_scaled_2d = scaler.fit_transform(X_train_2d)
        X_test_scaled_2d = scaler.transform(X_test_2d)
        
        X_train_scaled = X_train_scaled_2d.reshape(X_train.shape)
        X_test_scaled = X_test_scaled_2d.reshape(X_test.shape)
        
        # Handle imbalance dengan oversampling
        X_train_reshaped = X_train_scaled.reshape(X_train_scaled.shape[0], -1)
        from imblearn.over_sampling import RandomOverSampler
        ros = RandomOverSampler(random_state=SEED)
        X_train_bal_reshaped, y_train_bal = ros.fit_resample(X_train_reshaped, y_train)
        X_train_bal = X_train_bal_reshaped.reshape(-1, X_train_scaled.shape[1], X_train_scaled.shape[2])
        
        # Build and train simple model
        model = build_cnn_bilstm_attention_model(X_train_bal.shape[1:], num_classes)
        
        optimizer = Adam(learning_rate=0.001)
        model.compile(
            optimizer=optimizer,
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        history = model.fit(
            X_train_bal, to_categorical(y_train_bal, num_classes),
            validation_data=(X_test_scaled, to_categorical(y_test, num_classes)),
            epochs=30,
            batch_size=32,
            verbose=1,
            callbacks=[
                EarlyStopping(patience=10, restore_best_weights=True),
                ReduceLROnPlateau(factor=0.5, patience=5)
            ]
        )
        
        y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
        acc = accuracy_score(y_test, y_pred)
        
        print(f"\n✅ Simple model trained!")
        print(f"   Accuracy: {acc:.4f}")
        print(f"   F1-Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
        
        model.save('simple_cnn_bilstm_model.h5')
        print(f"   Model saved: simple_cnn_bilstm_model.h5")

print("\n" + "="*80)
print("🏁 PROGRAM FINISHED")
print("="*80)

c:\Program Files\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


TensorFlow Version: 2.20.0
NumPy Version: 1.26.4

🚀 LOADING DATA
⚠️  No processed data found, using sample data...
✅ Created sample data for testing

✅ Data ready:
  Total samples: 5000
  Sequence length: 1000
  Channels: 12
  Number of patients: 100
  Number of classes: 4
  Classes: ['Left_Ventricular_Region', 'Outflow_Tract_Region', 'Right_Ventricular_Region', 'Septal_Region']

🧬 CNN-BiLSTM-ATTENTION WITH ECG GAN FOR IMBALANCED DATA
⚠️  No GPU available, using CPU

🧪 CROSS-VALIDATION WITH ECG GAN


🎯 FOLD 1 / 3
  Train: 66 patients, 3318 beats
  Test: 34 patients, 1682 beats

⚡ Training with GAN augmentation...
  Input shape: (1000, 12)
  Generator output shape: (None, 1000, 12)
  Discriminator input shape: (None, 1000, 12)

🎨 Training ECG GAN for data augmentation...
  Epoch 0: D_loss = 0.6866, D_acc = 0.5938, G_loss = 0.6881
  Epoch 10: D_loss = 0.6931, D_acc = 0.3839, G_loss = 0.6829
  Epoch 20: D_loss = 0.6986, D_acc = 0.3732, G_loss = 0.6719
✅ ECG GAN training completed

🔧 Augme

KeyboardInterrupt: 